In [1]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None


def _find_env_file(start: Path) -> Path | None:
    candidates = [start, *start.parents]
    for folder in candidates:
        f = folder / ".env"
        if f.exists() and f.is_file():
            return f
    return None


env_file = _find_env_file(Path.cwd())
if env_file is not None:
    if load_dotenv is not None:
        load_dotenv(dotenv_path=env_file, override=False)
    else:
        for raw in env_file.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k and (k not in os.environ):
                os.environ[k] = v

# LLM: DeepSeek official
llm_api_key = os.getenv("DEEPSEEK_API_KEY")
llm_base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1")
llm_model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

# Embeddings: MoLi Ark (Gitee AI)
embed_api_key = os.getenv("MOLI_API_KEY")
embed_base_url = os.getenv("MOLI_BASE_URL", "https://ai.gitee.com/v1")
embed_model = os.getenv("MOLI_EMBEDDING_MODEL", "Qwen3-Embedding-4B")

if not llm_api_key:
    raise ValueError("Missing DEEPSEEK_API_KEY for LLM")
if not embed_api_key:
    raise ValueError("Missing MOLI_API_KEY for embeddings")


In [2]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(
    api_key=embed_api_key,
    base_url=embed_base_url,
    model=embed_model,
    default_headers={"X-Failover-Enabled": "true"},
    check_embedding_ctx_length=False,
    chunk_size=16,
)

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Keep the answer concise.

Question: {question}
Context: {context}
Answer:"""
)

llm = ChatOpenAI(
    api_key=llm_api_key,
    base_url=llm_base_url,
    model=llm_model,
    temperature=0,
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")


c:\Users\jd\.conda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


'{\n    "thoughts": {\n        "text": "The user is asking for a definition of \'Task Decomposition\' based on the provided context. The context includes a description of task decomposition as part of planning in AI agents, specifically referencing Chain of Thought and Tree of Thoughts.",\n        "reasoning": "The context directly explains Task Decomposition in the \'Component One: Planning\' section, so I can extract and summarize that information to answer the question concisely.",\n        "plan": "- Identify the relevant part of the context.\\n- Summarize the definition of Task Decomposition.\\n- Format the answer in the required JSON structure.",\n        "criticism": "I must ensure the answer is concise and directly addresses the question without adding extra information not present in the context.",\n        "speak": "Task Decomposition is a technique used in AI planning to break down complex tasks into smaller, manageable steps."\n    },\n    "command": {\n        "name": "ans